In [ ]:
{
 "cells": [],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 2
}

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

# 1. 데이터 로드 및 결측치 처리
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 2. 파생변수 19개 생성
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data

train = add_features(train)
test = add_features(test)

# 3. 인코딩
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']
for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])
    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen: le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

# KNN을 위한 스케일링 준비
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# 4. 1단계: 개별 모델 정의 (Level 1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'lgbm': LGBMRegressor(n_estimators=1200, learning_rate=0.08, num_leaves=127, reg_alpha=0.15, reg_lambda=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
    'knn': KNeighborsRegressor(n_neighbors=3, weights='distance'),
    'rf': RandomForestRegressor(n_estimators=500, max_depth=10, random_state=42, n_jobs=-1)
}

# 1단계 예측값을 담을 메타 피처 생성
meta_train = np.zeros((len(x_train), len(models)))
meta_test = np.zeros((len(x_test), len(models)))

print("=== 1단계: 베이스 모델 학습 시작 ===")
for i, (name, model) in enumerate(models.items()):
    print(f"[{name}] 학습 중...")
    test_preds = np.zeros(len(x_test))
    
    for tr_idx, va_idx in kf.split(x_train):
        if name == 'knn':
            X_tr, X_va = x_train_scaled[tr_idx], x_train_scaled[va_idx]
            test_data = x_test_scaled
        else:
            X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
            test_data = x_test
            
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        
        model.fit(X_tr, y_tr)
        meta_train[va_idx, i] = model.predict(X_va)
        test_preds += model.predict(test_data) / kf.n_splits
        
    meta_test[:, i] = test_preds
    print(f"[{name}] OOF MAE: {mean_absolute_error(y_train, meta_train[:, i]):.4f}")

# 5. 2단계: 메타 모델 학습 (Level 2)
print("\n=== 2단계: 메타 모델(Ridge) 학습 ===")
meta_model = Ridge(alpha=1.0)
meta_oof = np.zeros(len(x_train))
final_preds = np.zeros(len(x_test))

for tr_idx, va_idx in kf.split(meta_train):
    X_tr, X_va = meta_train[tr_idx], meta_train[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    
    meta_model.fit(X_tr, y_tr)
    meta_oof[va_idx] = meta_model.predict(X_va)
    final_preds += meta_model.predict(meta_test) / kf.n_splits

print(f"★ 최종 Stacking OOF MAE: {mean_absolute_error(y_train, meta_oof):.4f}")

# 6. 제출 파일 저장
os.makedirs('../submissions', exist_ok=True)
final_preds = np.clip(final_preds, 0, 1)
sample_submission['stress_score'] = final_preds
submit_path = '../submissions/submit_07_final_stacking.csv'
sample_submission.to_csv(submit_path, index=False)
print(f"★ 마지막 제출 파일 생성 완료: {submit_path}")

=== 1단계: 베이스 모델 학습 시작 ===
[lgbm] 학습 중...
[lgbm] OOF MAE: 0.1785
[knn] 학습 중...
[knn] OOF MAE: 0.2056
[rf] 학습 중...
[rf] OOF MAE: 0.2280

=== 2단계: 메타 모델(Ridge) 학습 ===
★ 최종 Stacking OOF MAE: 0.1802
★ 마지막 제출 파일 생성 완료: ../submissions/submit_07_final_stacking.csv
